This notebook fine tunes PI0.5 VLA model on my LeRobot [dataset](https://huggingface.co/datasets/ankithreddy/so101_pickplace_tools_v3) from hugging face

## 1 . Config


In [50]:
import logging
import os
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s")
log = logging.getLogger(__name__)

HF_TOKEN = os.environ["HF_TOKEN"]
WANDB_TOKEN = os.environ["WANDB_TOKEN"]


DATASET_ID = "ankithreddy/so101_pickplace_tools_v3"

CHECKPOINT_REPO = "ankithredd/pi05-so101-finetune"

RUN_STORAGE_PATH = "./checkpoints"
RUN_NAME         = "pi05-so101-pickplace-finetune"


## 2. Understanding the data

In [18]:
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
# from lerobot.datasets.streaming_dataset import StreamingLeRobotDataset

ds_meta = LeRobotDatasetMetadata(DATASET_ID)

log.info(f"dataset version: {ds_meta.info.get('codebase_version')}")
log.info(f"robot type    : {ds_meta.robot_type}")
log.info(f"total episodes: {ds_meta.total_episodes}")
log.info(f"total frames  : {ds_meta.total_frames}")
log.info(f"fps : {ds_meta.fps}")
log.info(f"cameras: {ds_meta.camera_keys}")
log.info(f"action dim: {ds_meta.features["action"]["shape"]}")
log.info(f"state dim: {ds_meta.features["observation.state"]["shape"]}")
log.info(f"tasks: {ds_meta.tasks}")

log.info("--- cameras ---")
for cam in ds_meta.camera_keys:
    log.info(f"camera {cam:<40} shape: {ds_meta.features[cam]['shape']}")

log.info("--- stats ---")
for key, stat in ds_meta.stats.items():
    log.info(f"  {key:<45} keys: {list(stat.keys())}")


2026-04-09 22:01:22,411  INFO      dataset version: v3.0
2026-04-09 22:01:22,412  INFO      robot type    : so101_follower
2026-04-09 22:01:22,412  INFO      total episodes: 90
2026-04-09 22:01:22,412  INFO      total frames  : 57019
2026-04-09 22:01:22,413  INFO      fps : 30
2026-04-09 22:01:22,413  INFO      cameras: ['observation.images.images.wrist', 'observation.images.images.top']
2026-04-09 22:01:22,413  INFO      action dim: (6,)
2026-04-09 22:01:22,413  INFO      state dim: (6,)
2026-04-09 22:01:22,415  INFO      tasks:                                        task_index
task                                             
pick up screwdriver and put it in box           0
2026-04-09 22:01:22,415  INFO      --- cameras ---
2026-04-09 22:01:22,416  INFO      camera observation.images.images.wrist          shape: (720, 1280, 3)
2026-04-09 22:01:22,416  INFO      camera observation.images.images.top            shape: (720, 1280, 3)
2026-04-09 22:01:22,417  INFO      --- stats ---
2026

## 3. Understandig the model 

In [35]:
from lerobot.policies.pi05.configuration_pi05 import PI05Config

config = PI05Config()
log.info(f"Model          : PI0.5 (PaliGemma + Action Expert)")
log.info(f"image res      : {config.image_resolution}")
log.info(f"chunk size     : {config.chunk_size}")
log.info(f"max state dim  : {config.max_state_dim}")
log.info(f"max action dim : {config.max_action_dim}")
log.info(f"normalization  : {config.normalization_mapping}")
log.info(f"dtype          : {config.dtype}")

2026-04-09 22:17:42,720  INFO      Metal backend detected, using mps.
2026-04-09 22:17:42,724  WARNING   Device 'None' is not available. Switching to 'mps'.
2026-04-09 22:17:42,727  INFO      Model          : PI0.5 (PaliGemma + Action Expert)
2026-04-09 22:17:42,728  INFO      image res      : (224, 224)
2026-04-09 22:17:42,729  INFO      chunk size     : 50
2026-04-09 22:17:42,729  INFO      max state dim  : 32
2026-04-09 22:17:42,730  INFO      max action dim : 32
2026-04-09 22:17:42,730  INFO      normalization  : {'VISUAL': <NormalizationMode.IDENTITY: 'IDENTITY'>, 'STATE': <NormalizationMode.QUANTILES: 'QUANTILES'>, 'ACTION': <NormalizationMode.QUANTILES: 'QUANTILES'>}
2026-04-09 22:17:42,730  INFO      dtype          : float32


### Observations

**Dataset**
- 90 episodes, 57,019 frames at 30fps
- 2 cameras: wrist + top, stored as `(720, 1280, 3)`, decoded to CHW float32
- State and action: 6-DOF joint positions
- Single task: "pick up screwdriver and put it in box"
- Stats: mean/std only — no quantile stats

**PI0.5**
- Accepts any image resolution, resizes to 224×224 internally
- State/action max dim: 32 — our 6-DOF fits, padded automatically
- Language conditioned — uses task string per sample
- Normalization default: QUANTILES — we override to MEAN_STD at training time

## 5. Connect to Ray

In [37]:
import ray

ray.init(
    runtime_env={
        "py_executable": "uv run",
        "working_dir": ".",
        "env_vars": {
            "HF_TOKEN": HF_TOKEN,
            "WANDB_API_KEY": WANDB_TOKEN,
        },
    },
    ignore_reinit_error=True,
)

log.info(ray.cluster_resources())

2026-04-09 22:26:30,071	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-04-09 22:26:35,942	INFO worker.py:2013 -- Started a local Ray instance.
2026-04-09 22:26:35,950	WARNING working_dir.py:86 -- Directory '.git' is now ignored by default when packaging the working directory. To disable this behavior, set the `RAY_OVERRIDE_RUNTIME_ENV_DEFAULT_EXCLUDES=''` environment variable.
2026-04-09 22:26:35,950	WARNING working_dir.py:86 -- Directory '.venv' is now ignored by default when packaging the working directory. To disable this behavior, set the `RAY_OVERRIDE_RUNTIME_ENV_DEFAULT_EXCLUDES=''` environment variable.
2026-04-09 22:26:35,953	INFO packaging.py:392 -- Ignoring upload to cluster for these files: [PosixPath('/Users/ankithreddy/Desktop/mlopsaiprojectsnres/new/vla-distributed-training/.gitignore')]
2026-04-09 22:26:35,958	INFO packaging.py:392 -- Ignoring upload to cluster for the

## 6. Ray Data: Building the pipeline

In [49]:
import importlib
import datasource
importlib.reload(datasource)
from datasource import LeRobotStreamingDatasource

source = LeRobotStreamingDatasource(DATASET_ID, revision="main")
ds = ray.data.read_datasource(source)

log.info(f"pipeline ready")
log.info(f"total episodes : {source.meta.total_episodes}")
log.info(f"cameras        : {source.meta.camera_keys}")

2026-04-09 22:53:42,225  INFO      pipeline ready
2026-04-09 22:53:42,225  INFO      total episodes : 90
2026-04-09 22:53:42,226  INFO      cameras        : ['observation.images.images.wrist', 'observation.images.images.top']


## 7. Ray Train - distributed training

In [ ]:
import torch

def train_loop_per_worker(config: dict):
    """
    Runs on each GPU worker. Ray Train launches N copies — one per GPU.
    Each worker gets its own shard of the Ray Data pipeline.

    Ray handles under the hood:
      - torch.distributed.init_process_group with NCCL
      - CUDA_VISIBLE_DEVICES per worker
      - Gradient sync across workers via DDP AllReduce
    """
    import ray.train
    import ray.train.torch
    from ray.air.integrations.wandb import setup_wandb
    from lerobot.policies.pi05 import PI05Policy
    from lerobot.policies.factory import make_pre_post_processors

    # Only rank 0 actually logs to wandb — other workers are silent no-ops
    wandb = setup_wandb(config, project=RUN_NAME)

    device = torch.device("cuda")

    # ── Load PI0.5 ────────────────────────────────────────────────────────────
    # train_expert_only=True freezes the PaliGemma backbone.
    # Only the 4 action/time heads are trained:
    # action_in_proj, action_out_proj, time_mlp_in, time_mlp_out
    policy = PI05Policy.from_pretrained(
        config["pretrained_path"],
        device=device,
        dtype=torch.bfloat16,
        train_expert_only=True,
        gradient_checkpointing=True,
    )

    # Freeze backbone, unfreeze only action heads — must happen BEFORE DDP wrap
    for p in policy.parameters():
        p.requires_grad = False
    for name, module in policy.model.named_children():
        if name in {"action_in_proj", "action_out_proj", "time_mlp_in", "time_mlp_out"}:
            for p in module.parameters():
                p.requires_grad = True

    # Wrap in DDP — Ray already called init_process_group with NCCL
    # After this, gradients are averaged across all GPUs automatically
    policy = ray.train.torch.prepare_model(policy)

    # ── Optimizer ─────────────────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(
        [p for p in policy.parameters() if p.requires_grad],
        lr=config["lr"],
    )
    scaler = torch.amp.GradScaler("cuda")

    # ── Resume from checkpoint ────────────────────────────────────────────────
    # Returns last checkpoint on failure recovery, None on fresh run
    checkpoint = ray.train.get_checkpoint()
    if checkpoint:
        import ray.cloudpickle as pickle
        with checkpoint.as_directory() as d:
            with open(f"{d}/state.pkl", "rb") as f:
                state = pickle.load(f)
        policy.module.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optim"])
        scaler.load_state_dict(state["scaler"])
        start_epoch, step = state["epoch"] + 1, state["step"]
    else:
        start_epoch, step = 0, 0

    # ── Normalization override ────────────────────────────────────────────────
    # PI0.5 defaults to QUANTILES — our dataset only has mean/std
    policy.module.config.normalization_mapping = {
        "ACTION": "MEAN_STD",
        "STATE":  "MEAN_STD",
        "VISUAL": "IDENTITY",
    }

    # Handles: state/action normalization, image resize to 224x224,
    # task string tokenization for PaliGemma
    preprocessor, _ = make_pre_post_processors(
        policy.module.config,
        pretrained_path=config["pretrained_path"],
        dataset_stats=config["stats"],
    )

    batch_size = config["batch_size"]
    grad_accum = config["grad_accum"]
    num_epochs = config["num_epochs"]

    # ── Training loop ─────────────────────────────────────────────────────────
    # Each worker gets its slice of the dataset automatically — no manual sharding
    shard = ray.train.get_dataset_shard("train")

    for epoch in range(start_epoch, num_epochs):
        optimizer.zero_grad(set_to_none=True)
        epoch_loss, epoch_count, accum_count = 0.0, 0, 0

        # Streams Arrow batches from Ray object store → torch tensors on GPU
        # CPU workers decoding next batch concurrently while GPU trains this one
        for batch in shard.iter_torch_batches(batch_size=batch_size, device=device):

            batch = preprocessor(batch)
            batch.pop("task", None)
            batch.pop("task_index", None)

            with torch.autocast("cuda", torch.bfloat16):
                out  = policy(batch)
                loss = out.loss if hasattr(out, "loss") else out[0]

            scaler.scale(loss / grad_accum).backward()
            step += 1
            accum_count += 1
            epoch_loss += float(loss.detach())
            epoch_count += 1

            if accum_count % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in policy.parameters() if p.requires_grad], 1.0
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                accum_count = 0

            if step % 10 == 0:
                log.info(f"epoch={epoch} step={step} loss={float(loss):.4f}")
                wandb.log({"loss": float(loss), "step": step})

        if accum_count > 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # ── Checkpoint + report ───────────────────────────────────────────────
        # Synchronization barrier — ALL workers must call ray.train.report()
        # Only rank 0 saves the checkpoint
        avg_loss = epoch_loss / max(epoch_count, 1)
        metrics  = {"epoch": epoch, "step": step, "loss": avg_loss}
        wandb.log({"epoch_loss": avg_loss, "epoch": epoch})

        if ray.train.get_context().get_world_rank() == 0:
            import tempfile, ray.cloudpickle as pickle
            ckpt_dir = tempfile.mkdtemp()
            with open(f"{ckpt_dir}/state.pkl", "wb") as f:
                pickle.dump({
                    "model":  policy.module.state_dict(),
                    "optim":  optimizer.state_dict(),
                    "scaler": scaler.state_dict(),
                    "epoch":  epoch,
                    "step":   step,
                }, f)
            ray.train.report(
                metrics,
                checkpoint=ray.train.Checkpoint.from_directory(ckpt_dir)
            )
        else:
            ray.train.report(metrics)

In [ ]:
import ray.train
import ray.train.torch

# ── Stats for normalization ───────────────────────────────────────────────────
# Extract mean/std from dataset metadata — passed into the preprocessor
# inside each GPU worker via config dict
stats = {
    k: {"mean": ds_meta.stats[k]["mean"], "std": ds_meta.stats[k]["std"]}
    for k in ("action", "observation.state")
}

# ── Training config ───────────────────────────────────────────────────────────
# Passed to every worker via train_loop_per_worker(config)
train_loop_config = {
    "pretrained_path": "lerobot/pi05_base",
    "stats":           stats,
    "total_rows":      ds_meta.total_frames,
    "num_epochs":      2,
    "batch_size":      2,      # per GPU — increase if you have VRAM headroom
    "grad_accum":      2,      # effective batch = batch_size * grad_accum * num_workers
    "lr":              1e-4,
}

# ── ScalingConfig ─────────────────────────────────────────────────────────────
# num_workers = number of GPUs
# On Mac: set use_gpu=False for testing (CPU only, no actual training)
# On DO/RunPod: set use_gpu=True and num_workers to however many GPUs you have
scaling_config = ray.train.ScalingConfig(
    num_workers=1,    # change to 2/4 on multi-GPU
    use_gpu=False,    # change to True on GPU machine
)

# ── RunConfig ─────────────────────────────────────────────────────────────────
# storage_path: where checkpoints are saved
# max_failures=1: if a worker dies, Ray restarts and resumes from last checkpoint
run_config = ray.train.RunConfig(
    name=RUN_NAME,
    storage_path=RUN_STORAGE_PATH,
    failure_config=ray.train.FailureConfig(max_failures=1),
)

# ── TorchTrainer ──────────────────────────────────────────────────────────────
# Single entry point for distributed PyTorch on Ray.
# To scale: just change num_workers — nothing else changes.
trainer = ray.train.torch.TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config=train_loop_config,
    scaling_config=scaling_config,
    run_config=run_config,
    datasets={"train": ds},
)

result = trainer.fit()
log.info(f"training complete: {result}")